# 03 · Transfer learning

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = ""
USE_DRIVE = True

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub", "scikit-learn", "scipy", "pandas", "matplotlib", "pillow", "tqdm"], check=True)
    if USE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        os.environ.setdefault("XRAY_ARTIFACTS", "/content/drive/MyDrive/xray_project/artifacts")
        os.environ.setdefault("XRAY_RESULTS", "/content/drive/MyDrive/xray_project/results")


def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "config.py").exists():
            return p


ROOT = find_root()
if ROOT is None:
    assert REPO_URL, "Set REPO_URL or open the notebook from inside the repository"
    subprocess.run(["git", "clone", REPO_URL, "xray-vision-pneumonia-CNN"], check=True)
    ROOT = Path("xray-vision-pneumonia-CNN").resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config as C
from src import evaluate as E
from src import models as M
from src import train as T
from src.dataset import AugConfig
from src.utils import load_runs, sync_results_to_repo

C.ensure_dirs()
ARCHS = list(M.TL_ARCHS)
TL_SEEDS = [C.SEED]
DEFAULT_FT = "ft-u30-lr1e-05"

## Frozen head and fine-tuned models, shared recipe (class weights + augmentation)

In [ ]:
for arch in ARCHS:
    for seed in TL_SEEDS:
        T.train_transfer(arch, seed)

In [ ]:
runs = load_runs()
is_main = runs["family"].isin(ARCHS) & runs["tag"].isna() & (runs["strategy"] == "cw_aug")
main = runs[is_main & ((runs["variant"] == "frozen") | runs["run_id"].str.contains(DEFAULT_FT, regex=False))]
full = E.metrics_table(main["run_id"].tolist(), "val").merge(main[["run_id", "family", "variant", "seed"]], on="run_id")
full.to_csv(C.RESULTS_DIR / "val_metrics_transfer.csv", index=False)
full[["family", "variant", "seed", "accuracy", "precision", "sensitivity", "specificity", "f1", "roc_auc", "pr_auc", "spec_at_target_sens"]].round(4)

In [ ]:
for arch in ARCHS:
    for seed in TL_SEEDS:
        base = f"{arch}-cw_aug-s{seed}"
        T.plot_transfer_curves(f"{base}-frozen", f"{base}-{DEFAULT_FT}", C.FIG_DIR / f"curves_{arch}_transfer.png")

In [ ]:
scratch = runs[(runs["family"] == "scratch") & runs["tag"].isna() & (runs["strategy"] == "cw_aug")]
compare = pd.concat([scratch, main])[["run_id", "family", "variant", "seed", "val_auc", "val_pr_auc", "val_sensitivity", "val_specificity", "val_spec_at_target_sens"]]
compare.round(4).to_csv(C.RESULTS_DIR / "compare_scratch_vs_transfer.csv", index=False)
compare.round(4)

## Ablations on the best architecture (seed 42)

In [ ]:
ft = main[main["variant"] == "finetune"]
BEST_ARCH = ft.sort_values("val_auc").iloc[-1]["family"]
print("Best architecture by validation ROC-AUC:", BEST_ARCH)

In [ ]:
for n_unfrozen in (30, 60, None):
    T.train_transfer(BEST_ARCH, C.SEED, n_unfrozen=n_unfrozen)
for lr in (1e-4, 3e-5, 1e-5):
    T.train_transfer(BEST_ARCH, C.SEED, ft_lr=lr)
T.train_transfer(BEST_ARCH, C.SEED, aug=AugConfig(hflip=True), tag="hflip")
T.train_transfer(BEST_ARCH, C.SEED, aug=AugConfig(brightness=0, contrast=0), tag="noBC")
T.train_transfer(BEST_ARCH, C.SEED, resize="pad", tag="pad")

In [ ]:
runs = load_runs()
abl = runs[(runs["family"] == BEST_ARCH) & (runs["variant"] == "finetune") & (runs["seed"] == C.SEED) & (runs["strategy"] == "cw_aug")].copy()
abl["n_unfrozen"] = [json.loads(p)["n_unfrozen"] for p in abl["params"]]
abl["lr"] = [json.loads(p)["lr"] for p in abl["params"]]
show = ["run_id", "tag", "n_unfrozen", "lr", "epochs_trained", "val_auc", "val_pr_auc", "val_sensitivity", "val_specificity", "val_spec_at_target_sens"]
abl = abl[show].round(5)
abl.to_csv(C.RESULTS_DIR / "ablation_transfer.csv", index=False)
abl

## Notes

Grayscale input is replicated to three channels inside the model (`GrayToRGB`) and passed through each architecture's own ImageNet preprocessing. Batch-normalisation layers stay in inference mode during fine-tuning. The `hflip` run measures whether horizontal flipping (which mirrors the heart) helps; `noBC` removes brightness/contrast augmentation; `pad` keeps the aspect ratio instead of squashing images to a square.

In [ ]:
sync_results_to_repo()